# Домашнее задание 2 (10 баллов).

*Все задания ниже имеют равный вес*

Код для импорта мы написали за вас (не благодарите, нам не трудно). Дальше код будете писать вы. 

[Тут](https://habr.com/ru/companies/ruvds/articles/494720/) шпора по pandas. За основу домашнего задания взят ноутбук [отсюда](https://rutube.ru/video/f884aa6ed5f94120b7304506042fe5bb/) (не подглядывайте!).

In [1]:
import pandas as pd
import numpy as np

#### Описание данных

Автор д/з - плохой человек, который не стал переводить описание с мотивировкой, что весь DS на английском. Так что описание полей будет на английском:

1. Account ID
- Description: A unique identifier for each social media account in the dataset.
- Type: Integer
- Example: 1, 2, 3, …
2. Username
- Description: The username or handle of the social media account.
- Type: String
- Example: john_doe, tech_guru_22, fitness_freak
3. Platform
- Description: The social media platform the account is using (Instagram, Twitter, Facebook, TikTok, LinkedIn).
- Type: Categorical (String)
- Example: Instagram, Twitter, Facebook, TikTok, LinkedIn
4. Follower Count
- Description: The total number of followers the account has.
- Type: Integer
- Example: 1500, 245000, 78000
5. Posts Per Week
- Description: The average number of posts the account creates per week.
- Type: Integer
- Example: 3, 5, 7
6. Engagement Rate
- Description: The percentage of interactions (likes, comments, shares) relative to the follower count. This is a measure of how engaging the content is.
- Type: Float
- Range: 0.01 to 0.15
- Example: 0.045 (4.5% engagement rate)
7. Ad Spend (USD)
- Description: The monthly amount spent on advertising or promoting posts.
- Type: Float
- Example: 150.75, 850.00, 300.50
8. Conversion Rate
- Description: The percentage of users who take a desired action (e.g., clicking a link, signing up, etc.) after interacting with an ad.
- Type: Float
- Range: 0.01 to 0.05 (1% to 5% conversion rate)
- Example: 0.025 (2.5% conversion rate)
9. Campaign Reach
- Description: The total number of unique users reached by the user’s campaigns in a given month.
- Type: Integer
- Example: 5000, 20000, 15000

#### Задание 0

Подгрузите данные. Да-да, за чтение таблицы баллов не будет))

**Hint**: [pd.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)

In [2]:
df = pd.read_csv("data.csv", sep=",", index_col=0)

In [3]:
#df = pd.read_csv("data.csv")

#### Задание 1

Колонка `Platform` содержит название различных платформ. Давайте представим, что в них есть некоторое отношение порядка. Закодируйте каждую платформу целым числом (от 0 до N) и положите этот "код" в новую колонку `Platform_Code`. Теперь вычислите корреляцию Спирмена между всеми парами колонок в датасете (результатом будет таблица корреляций). В качестве ответа выведите значение корреляции `Platform_Code` с `Engagement Rate`. Можете после вывода числа еще коротко написать, что оно означает (нет, это не оценивается).

**Hint**: [pd.factorize](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.factorize.html), [pd.DataFrame.select_dtypes](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.select_dtypes.html), [pd.DataFrame.corr](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html).

In [4]:
# ( ੭ ･ᴗ･ )੭
# Кодируем платформы
codes, uniques = pd.factorize(df["Platform"])
df["Platform_Code"] = codes

# Считаем корреляцию
numeric_part = df.select_dtypes(include="number")
spearman_corr = numeric_part.corr("spearman") # таблица корреляций
print(spearman_corr["Platform_Code"]["Engagement Rate"]) # ответ

0.03138169529349812


#### Задание 2

Теперь посмотрите на столбец `Follower Count`. В нем какие-то числа. Иногда бывает полезно провести дискретизацию такого признака. Разбейте все значения в столбце на 4 группы: "Low", "Medium", "High", "Very High". Каждая группа включает в себя новые 25% данных. То есть, Low включает в себя 25% самых маленьких значений признака и так далее. Положите значения "Low", "Medium", "High" или "Very High" для каждого сэмпла датасета в новую колонку `Follower_Bin`. Теперь посчитайте среднее значение `Engagement Rate` для каждой категории из `Follower_Bin`. В качестве ответа выведите значение для категории "High".

**Hint**: [pd.qcut](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.qcut.html), [pd.groupby](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html), [pd.DataFrame.mean](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html)

In [5]:
# (◕^^◕)
# Разделяем по группам и кладём в колонку
df["Follower_Bin"] = pd.qcut(df["Follower Count"], [0, .25, .5, .75, 1.], ["Low", "Medium", "High", "Very High"])
# Группируем и считаем среднее
means_in_categories = df.groupby(["Follower_Bin"], observed=True)["Follower Count"].mean()
print(means_in_categories["High"]) # ответ


628407.5112


#### Задание 3

Иногда бывает полезно превратить широкую таблицу в длинную (например, для визуализаций сразу нескольких признаков на одной картинке). Да, звучит странно, но именно этим вы сейчас и займетесь. Сделайте новый датафрейм `melted_df`, в который вы поместите каждый сэмпл датасета 6 раз: по одному разу на значение из 'Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate' и 'Campaign Reach'. То есть, вы берете сэмпл из датасета (строку) и превращаете ее в 6 отдельных строк. Каждая отдельная строка в столбце `Metric` имеет имя из предложенного списка 5 признаков, а в столбце `Value` - значение данного сэмпла по этому признаку. Значение `Platform` повторяется в этих 6 строках.

Иначе говоря, 

```json
{
    "Account ID": 1,
    "Username": "harrislisa",
    "Platform": "TikTok",
    "Follower Count": 54217,
    "Posts Per Week": 3,
    "Engagement Rate": 0.0986,
    "Ad Spend (USD)": 538.1,
    "Conversion Rate": 0.049,
    "Campaign Reach": 1308,
    "Platform_Code": 0,
    "Follower_Bin": "Low"
}
```

превращается в 

```json
{
    "Platform": "TikTok",
    "Metric": "Follower Count",
    "Value": 54217,
},
{
    "Platform": "TikTok",
    "Metric": "Posts Per Week",
    "Value": 3,
}, ...
```

Для каждого уникальной пары значений (`Platform`, `Metric`) посчитайте моду среди всех значений `Value` для этой пары, результат сделайте списком и оставьте только наибольшее. В качестве ответа выведите сумму полученных мод (сумму всех значений в столбце `Value` уже после вычисления мод). Иначе говоря, выведите сумму всех мод значений для всех уникальных пар (`Platform`, `Metric`).

**Hint**: [pd.melt](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.melt.html), [pd.DataFrame.mode](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mode.html), [pd.DataFrameGroupBy.agg](https://pandas.pydata.org/docs/dev/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)

In [ ]:
""" (づ๑•ᴗ•๑)づ♡
Ссылки на функции и методы:
transform - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.transform.html
"""
# melt не дал такого результата как в примере (сначала 6 строк от первой строки, потом 6 от второй и т.д.) но в целом вышел такой-же результат, только порядок строк другой
melted_df = pd.melt(df, id_vars=["Platform"], value_vars=['Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate', 'Campaign Reach'], var_name="Metric", value_name="Value")

# группируем данные + считаем моду спиком
mode_series = melted_df.groupby(["Platform", "Metric"])["Value"].agg(lambda x: x.mode().tolist())

# оставляем наибольшее и делаем результат списком
mode_list = mode_series.transform(lambda x: max(x)).tolist() # результат
# сумма всех мод значений (именно максимальных, т.е. тех, которые остались в списке)
print(sum(mode_list)) # ответ

# сумма вообще всех мод (если вдруг я неправильно поняла задание)
#print(sum(mode_series.transform(lambda x: sum(x)).tolist()))


3100285.4716


#### Задание 4

А теперь хочется посмотреть на самые популярные аккаунты на разных платформах. Для каждой платформы отсортируйте датафрейм по убыванию количества подписчиков (`Follower Count`) - да, без циклов, сразу для всех платформ сделать сортировку, а затем оставьте только первые три записи для каждой платформы - это и будут три самых популярных аккаунта для каждой платформы. В качестве ответа выведите саму таблицу и минимальное значение `Follower Count` в ней.

**Hint**: к *groupby* можно применять функции - это эквивалентно применению функции к каждой "группе" внутри groupby-объекта. Читайте [про применение apply к датафрейму после groupby](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#flexible-apply).

In [ ]:
# ε(´סּ︵סּ`)з
# группируем, сортируем и оставляем первые 3 записи в каждой группе
platform_groups_sorted = df.groupby(["Platform"]).apply(lambda x: x.sort_values("Follower Count", ascending=False).head(3), include_groups=False)
print(platform_groups_sorted) # таблица
print()
print(platform_groups_sorted["Follower Count"].min()) # минимальное значение Follower Count в новой таблице

                             Username  Follower Count  Posts Per Week  \
Platform  Account ID                                                    
Facebook  2404                 eric65          999982               6   
          7351           patricknoble          997915               3   
          1690            chavezjason          997512               7   
Instagram 8686        alexandersamuel          999726               3   
          3966               lrodgers          999351               1   
          2190                 jbrown          997844               5   
LinkedIn  3040                toneill          999055               4   
          6360          andrewgregory          998968               7   
          2160           ashleycooper          998925               6   
TikTok    5839           edwardthomas          999739               7   
          4235          andradewesley          999234               5   
          2576           williamwyatt          9986

#### Задание 5

Хочется посчитать какую-то метрику. Мы хотим посмотреть, на отношение разности суммы подписчиков аккаунтов с высокой и низкой конверсией к суммарному охвату рекламы на каждой платформе. То есть, мы делим аккаунты на две группы: высокая и низка конверсия. Затем мы смотрим на то, на сколько сильно влияние аккаунтов с высокой конверсией по сравнению с аккаунтами с низкой конверсией. 

Давайте определим *Conversion Influence* следущим образом:

$$Conversion Influence = \frac{Total Follower\ Count (High) - Total Follower\ Count (Low)}{Total Campaign Reach (High)+Total Campaign Reach (Low)}$$

Считать эту метрику мы будет для каждой `Platform`. В этой формуле High - это значения всех сэмплов датасета, в которых `Conversion Rate` больше медианы, а `Low` - не более медианы. `Total Feature` - это суммарное количество значений `Feature` либо по `High` сэмплам, либо по `Low`.

Чтобы постоянно не пересчитывать, где High. где Low, сделайте новую колонку в датасете `Conversion_Category`. Положите в нее для каждой строки либо High, либо Low.

Выведите платформу с самым большим `Conversion Influence`.

**Hint**: данное задание не про *groupby*, а скорее про [pd.pivot_table](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html). Сделайте сводную таблицу, по которой уже можно посчитать суммы, а затем подставить их в формулы.

In [ ]:
""" (︶ω︶)
Ссылки на функции и методы:
median - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.median.html
where - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.where.html
"""
# считаем медиану
conversion_rate_median = df["Conversion Rate"].median()

# ставим значения в столбец
df["Conversion_Category"] = "Low"
df["Conversion_Category"] = df["Conversion_Category"].where(df["Conversion Rate"] <= conversion_rate_median, "High")

# группируем + считаем суммы
total_follower_count_table = pd.pivot_table(df, values="Follower Count", index=["Platform"], columns=["Conversion_Category"], aggfunc="sum")
total_campaign_reach_table = pd.pivot_table(df, values="Campaign Reach", index=["Platform"], columns=["Conversion_Category"], aggfunc="sum")

# считаем формулу для каждой платформы
results_conversion_influence = pd.DataFrame()
results_conversion_influence["Conversion Influence"] = (total_follower_count_table["High"] - total_follower_count_table["Low"]) / (total_campaign_reach_table["High"] + total_campaign_reach_table["Low"])

print(results_conversion_influence["Conversion Influence"].idxmax()) # ответ

Twitter


#### Задание 6

Мы знаем, что вам понравилось считать метрики по формуле. Давайте закрепим этот успех. Теперь для каждой платформы посчитаем, на сколько эффективна реклама в разрезе трех последовательных записей в датасете. 

Для каждой платформы отсортируйте записи в порядке убывания `Posts Per Week`. Будто бы аккаунты, которые постят чаще, используют более "активные" стратегии по рекламе. Теперь посчитайте *скользущие суммы с окном 3* по `Campaign Reach` и `Ad Spend (USD)`. Скользящая сумма с окном N - это вы идете по массиву, берете все последовательные тройки записей и суммируете их. Для первых двух записей троек не найдется. Для них скользящее среднее - NaN, что нам не помешает. 

Теперь для каждого окна посчитайте 

$$Rolling Efficiency Ratio = \frac{Rolling Sum of Campaign Reach}{Rolling Sum of Ad Spend}$$

По сути, для каждого окна вы посчитаете сколько пользователе привлеклось за один доллар, потреченный на рекламу, в данном окне. Понятно, что значений будет столько, сколько окон. Нам интересно максимально значение такой эффективности для каждой платформы.

В качестве ответа выведите название платформы с наибольшей максимальной эффективность и наименьшей (два названия, не одно, не три, ровно два).

**Hint**: окна можно делать через [pd.DataFrame.rolling](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html).

In [62]:
""" (◔/‿\◔)
Ссылки на функции (и не только):
get_level_values - https://pandas.pydata.org/docs/reference/api/pandas.Index.get_level_values.html
values - https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.values.html
"""
# группируем + сортируем
platform_groups_sorted_by_posts = df.groupby(["Platform"]).apply(lambda x: x.sort_values("Posts Per Week", ascending=False), include_groups=False)

# считаем сколзящие суммы
group_sum_by_campaign = platform_groups_sorted_by_posts.groupby(["Platform"])["Campaign Reach"].rolling(window=3).sum()
group_sum_by_ad_spend = platform_groups_sorted_by_posts.groupby(["Platform"])["Ad Spend (USD)"].rolling(window=3).sum()

# считаем формулу для каждого окна
results_rolling_efficiency = pd.DataFrame(index=group_sum_by_campaign.index.get_level_values(1))
results_rolling_efficiency["Rolling Efficiency Ratio"] = group_sum_by_campaign.values / group_sum_by_ad_spend.values

# смотрим максимальное значение формулы для каждой платформы
results_rolling_efficiency = results_rolling_efficiency.groupby(["Platform"])["Rolling Efficiency Ratio"].apply(lambda x: x.max())

# название платформы с наибольшей макс и с наименьшей макс
print(results_rolling_efficiency.idxmax()) # максимальная эффективность
print(results_rolling_efficiency.idxmin()) # минимальная эффективность


Facebook
LinkedIn


#### Задание 7

Это еще не все прекрасные функции pandas, которые мы хотим вам показать. Теперь вы посчитаете, сколько аккаунтов на каждой платформе одновременно лучшие по `Engagement Rate` и `Conversion Rate`.

Сделайте два отдельных суб-сета. В одном оставьте для каждой платфмормы один топовый аккаунт по `Engagement Rate`, в другом - по `Conversion Rate`. Соедините эти два подмножества по столбцу `Platform` так, что в одно строке есть описание сразу двух аккаунтов-лидеров. Теперь посмотрите равны ли имена аккаунтов в одной строке. Выведите количество строк, в которых названия аккаунтов совпадают.

In [63]:
""" ( ͡° ͜ʖ ͡°)
Ссылки на функции и методы:
to_frame - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.to_frame.html
to_list - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.to_list.html
"""
# максимумы
max_engagement = df["Engagement Rate"].max()
max_conversion = df["Conversion Rate"].max()

# суб-сет 1: группировка по платформам + топовые акки по Engagement
# Условие неоднозначное, я поняла как в первом абзаце: "... посчитате, сколько аккаунтов на каждой платформе одновременно лучшие"
# + кто-то из ассистов в чате предложил реализовать подход, где лидер может быть не только один, поэтому вот так
group_top_engagemet_by_platform = df.groupby(["Platform"]).apply(lambda x: set(x[x["Engagement Rate"] == max_engagement]["Username"].to_list()), include_groups=False).to_frame(name='Top Engagement Username')

# суб-сет 2: группировка по платформам + топовые акки по Conversion
group_top_conversion_by_platform = df.groupby(["Platform"]).apply(lambda x: set(x[x["Conversion Rate"] == max_conversion]["Username"].to_list()), include_groups=False).to_frame(name='Top Conversion Username')

# соединение по столбцу
merged_table = pd.merge(group_top_engagemet_by_platform, group_top_conversion_by_platform, on="Platform")

# табличка с количеством лидеров для каждой платформы
result_account_leaders_table = merged_table.apply(lambda x: len(x["Top Engagement Username"] & x["Top Conversion Username"]), axis=1).to_frame(name='Top Username Count')

# выводим суммарное количество лидеров
print(result_account_leaders_table["Top Username Count"].sum())

6


#### Задание 8

Давайте теперь что-то попроще сделаем. Например, посчитаем отношение суммарного количества подписчиков на аккаунтах с высокой конверсией к такой же сумме в аккаунтах с низкой конверсией (очевидно, для каждой платформы). По сути, мы просто хотим получить число, которое характеризует, на сколько сильно аккаунты с высокой конверсией "доминируют" над аккаунтами с низкой конверсией в плане количества подписчиков.

Высокой конверсией будем считать конверсию больше средней. Остальное - низкая. Посчитайте суммы подписчиков для каждой платформы, поделите одно на другое и выведите разницу между самым большим значением и самым маленьким, а также платформы, которые соотвутствуют этим значениям.

Используйте магическую команду `%%time`, чтобы замерить, сколько времени ушло на исполнение вашего pandas-скрипта.

In [ ]:
%%time
""" (◡‿◡✿)
Ссылки на функции и методы:
loc - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.loc.html
"""
# средняя конверсия (по всей таблице)
mean_conversion = df["Conversion Rate"].mean()
# делим на платформы + на низкую/высокую конверсию + суммируем подписочников по конверсиям
group_by_platform_high_conv = df.groupby(["Platform"]).apply(lambda x: x.loc[x["Conversion Rate"] > mean_conversion, "Follower Count"].sum(), include_groups=False)
group_by_platform_low_conv = df.groupby(["Platform"]).apply(lambda x: x.loc[x["Conversion Rate"] <= mean_conversion, "Follower Count"].sum(), include_groups=False)

# новая табличка: делим одно на другое по платформам
group_conversion_relation_by_platform = pd.DataFrame(index=group_by_platform_low_conv.index)
group_conversion_relation_by_platform["Conversion Relaton"] = group_by_platform_high_conv.values / group_by_platform_low_conv.values

# мне показалось что здесь много данных на выходе, было бы неплохо подписать так
# выводим разницу между самым большим и самым маленьким
print(f"Разница между самым большим и самым маленьким значением: {group_conversion_relation_by_platform["Conversion Relaton"].max() - group_conversion_relation_by_platform["Conversion Relaton"].min()}")

# выводим самое большое и самое маленькое
print(f"Платформа с самым большим значением: {group_conversion_relation_by_platform["Conversion Relaton"].idxmax()}")
print(f"Платформа с самым маленьким значением: {group_conversion_relation_by_platform["Conversion Relaton"].idxmin()}")


Разница между самым большим и самым маленьким значением: 0.17688741338715763
Платформа с самым большим значением: Twitter
Платформа с самым маленьким значением: Instagram
CPU times: total: 15.6 ms
Wall time: 22.7 ms


#### Задание 9

А теперь решите задание 8 чисто питоном. Никаких функций и методов pandas. Только питоновские циклы. Замерьте время выполнения кода. Наконец, сравните время в задании 8 и 9. Напишите ниже, кто же победил: чистый python и pandas?

**Hint**: Чтобы итерироваться по датафрейму, можно из него сделать генератор через [pd.DataFrame.iterrows](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iterrows.html) или [pd.DataFrame.itertuples](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.itertuples.html#pandas.DataFrame.itertuples). К слову, это не все способы итерироваться по датафрейму.

In [46]:
%%time
# (✿◠‿◠)
# посчитать среднее проходом по таблице
mean_conversion_py = 0 # не списочными выражениями, чтобы не проходить 2 раза по итератору
count_rows = 0
for row in df.itertuples():
    mean_conversion_py += row._7
    count_rows += 1
mean_conversion_py /= count_rows
mean_conversion_py
# делим на платформы + суммируем подписчиков
platforms_high_conv_sum = {}
platforms_low_conv_sum = {}
#3
for row in df.itertuples():
    if row.Platform not in platforms_high_conv_sum:
        platforms_high_conv_sum[row.Platform] = 0
        platforms_low_conv_sum[row.Platform] = 0
    if row._7 > mean_conversion_py:
        platforms_high_conv_sum[row.Platform] += row._3
    else:
        platforms_low_conv_sum[row.Platform] += row._3

# считаем формулу
platforms_conv_relation = {}
for key in platforms_high_conv_sum.keys():
    platforms_conv_relation[key] = platforms_high_conv_sum[key] / platforms_low_conv_sum[key]

#тоже всё подписано
# выводим разницу между самым большим и самым маленьким
max_key = max(platforms_conv_relation, key=platforms_conv_relation.get)
min_key = min(platforms_conv_relation, key=platforms_conv_relation.get)
print(f"Разница между самым большим и самым маленьким значением: {platforms_conv_relation[max_key] - platforms_conv_relation[min_key]}")

# выводим самое большое и самое маленькое
print(f"Платформа с самым большим значением: {max_key}")
print(f"Платформа с самым маленьким значением: {min_key}")

Разница между самым большим и самым маленьким значением: 0.17688741338715763
Платформа с самым большим значением: Twitter
Платформа с самым маленьким значением: Instagram
CPU times: total: 78.1 ms
Wall time: 68.4 ms


**А победителем является**: pandas

#### Задание 10

Крайне серьезное задание. Отнеситесь к нему соответствующе. В ячейке ниже напишите ваш любимый анекдот или мем (только без баянов, окей?). Можно плохие. Помните, это задание на полный балл. Проверяющий работу ассистент должен улыбнуться.

Если вставляете картинку, то убедитесь, что вы ее не подгружаете локально. А то будет неудобно - потерять балл на этом задании, когда надо было выложить картинку на облако и прокинуть ссылку. И нет, нельзя сюда просто ссылку вставить. Либо ищите, как вставить картинку, либо смешной анекдот. Есть всего два стула - выбирайте...

In [60]:
# ‿( ́ ̵ _-`)‿
from IPython.display import HTML

HTML('<img src="https://drive.google.com/file/d/1P17tHlHOg18zBeeE-ScAVdEw_EuUjPjD/view?usp=sharing" width="400">')
#У меня картинка почему-то не загрузилась, поэтому анекдот бонусом: 
#Штирлиц едет по дороге на автомобиле. На обочине стоит Мюллер - голосует.
#"Хрен тебе!" - подумал Штирлиц, поддал газу и проехал мимо.

#Через некоторое время Штирлиц опять видит голосующего Мюллера.
#"Хрен тебе!" - подумал Штирлиц, поддал газу и проехал мимо.

#Через некоторое время Штирлиц опять видит голосующего Мюллера.
#"Хрен тебе!" - подумал Штирлиц, поддал газу и проехал мимо.

#Через некоторое время Штирлиц опять видит голосующего Мюллера.
#"Издевается!" - подумал Мюллер. "Кольцевая!" - подумал Штирлиц
